In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

# Load a subset of the data
df = pd.read_csv('Reviews.csv').head(10000)

# 1. Clean missing values and duplicates
df.dropna(subset=['Text', 'Score'], inplace=True)
df.drop_duplicates(subset=['Text'], inplace=True)

# 2. Label Encoding: Convert stars to sentiment
def set_sentiment(score):
    if score <= 2: return 0 # Negative
    elif score == 3: return 1 # Neutral
    else: return 2 # Positive

df['Sentiment'] = df['Score'].apply(set_sentiment)

# 3. Text Preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text) # Remove HTML tags
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    return text

df['Cleaned_Text'] = df['Text'].apply(clean_text)

# 4. Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    df['Cleaned_Text'], df['Sentiment'], test_size=0.2, random_state=42
)

In [ ]:
# Step 4: Exploratory Data Analysis (EDA)

import matplotlib.pyplot as plt
import pandas as pd

# Basic Statistical Summary
print("Statistical Summary of Scores:")
print(df['Score'].describe())
print("\nSentiment Distribution:")
print(df['Sentiment'].value_counts())

# 1. Histogram - Distribution of Review Scores
plt.figure()
plt.hist(df['Score'], bins=5)
plt.title("Distribution of Review Scores")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.show()

# 2. Bar Chart - Sentiment Distribution
plt.figure()
df['Sentiment'].value_counts().sort_index().plot(kind='bar')
plt.title("Sentiment Class Distribution (0=Negative, 1=Neutral, 2=Positive)")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

# 3. Boxplot - Review Length vs Sentiment
df['Review_Length'] = df['Cleaned_Text'].apply(lambda x: len(x.split()))

plt.figure()
plt.boxplot([
    df[df['Sentiment'] == 0]['Review_Length'],
    df[df['Sentiment'] == 1]['Review_Length'],
    df[df['Sentiment'] == 2]['Review_Length']
])
plt.title("Review Length Distribution by Sentiment")
plt.xlabel("Sentiment (0=Neg, 1=Neu, 2=Pos)")
plt.ylabel("Review Length (Word Count)")
plt.show()

# 4. Scatter Plot - Review Length vs Score
plt.figure()
plt.scatter(df['Review_Length'], df['Score'])
plt.title("Review Length vs Review Score")
plt.xlabel("Review Length (Word Count)")
plt.ylabel("Score")
plt.show()

# 5. Correlation Heatmap (Numerical Features Only)
numerical_df = df[['Score', 'Sentiment', 'Review_Length']]
correlation_matrix = numerical_df.corr()

plt.figure()
plt.imshow(correlation_matrix)
plt.title("Correlation Heatmap")
plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=45)
plt.yticks(range(len(correlation_matrix.columns)), correlation_matrix.columns)
plt.colorbar()
plt.show()

print("\nCorrelation Matrix:")
print(correlation_matrix)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# ==============================
# 1. Remove Irrelevant Features
# ==============================

# Keep only necessary columns
df_model = df[['Cleaned_Text', 'Sentiment', 'Review_Length']]

# ==============================
# 2. Convert Text to Numerical (TF-IDF)
# ==============================

tfidf = TfidfVectorizer(
    max_features=5000,      # limit features to avoid overfitting
    stop_words='english',   # remove common words
    ngram_range=(1,2)       # use unigrams + bigrams
)

X_text = tfidf.fit_transform(df_model['Cleaned_Text'])

# ==============================
# 3. Scale Numerical Feature
# ==============================

scaler = StandardScaler()
X_length = scaler.fit_transform(df_model[['Review_Length']])

# Convert scaled length to sparse format and combine
from scipy.sparse import hstack

X_final = hstack((X_text, X_length))

y = df_model['Sentiment']

# ==============================
# 4. Train-Test Split
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

print("Final Feature Shape:", X_final.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# 1. Model Choice Rationale
# Logistic Regression is highly effective for high-dimensional sparse data
# produced by combining TF-IDF and scaled numerical features.

# 2. Define the Model
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# 3. Hyperparameter Tuning (Grid Search)
# We tune 'C' (regularization) and 'solver' to find the best fit
param_grid = {
    'C': [0.1, 1, 10],
    'solver': ['liblinear'] # liblinear is robust for small to medium datasets
}

grid_search = GridSearchCV(
    estimator=lr_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1
)

# 4. Training the Model
print("Training model on combined TF-IDF and Review Length features...")
grid_search.fit(X_train, y_train)

# 5. Extract the best model
best_model = grid_search.best_estimator_

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Validation Accuracy: {grid_search.best_score_:.4f}")


Training model on combined TF-IDF and Review Length features...
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best Parameters: {'C': 10, 'solver': 'liblinear'}
Best Validation Accuracy: 0.8267


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Generate Predictions
y_pred = best_model.predict(X_test)

# 2. Calculate Metrics
accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

# 3. Create Performance Report Table
performance_metrics = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Value': [accuracy, precision, recall, f1]
}
df_perf = pd.DataFrame(performance_metrics)
print("--- Performance Report ---")
print(df_perf)
print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

# 4. Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])
plt.title('Confusion Matrix: Predicted vs Actual Sentiment')
plt.ylabel('Actual Sentiment')
plt.xlabel('Predicted Sentiment')
plt.show()

In [ ]:
import numpy as np

# Get feature names from the TF-IDF vectorizer
feature_names = np.array(tfidf.get_feature_names_out())

# Get the coefficients for the 'Negative' class (index 0) and 'Positive' class (index 2)
# Since we added 'Review_Length' as the last feature, we handle it separately
coef_neg = best_model.coef_[0]
coef_pos = best_model.coef_[2]

# Find top 10 words for Negative and Positive
top_neg = feature_names[np.argsort(coef_neg[:-1])[-10:]]
top_pos = feature_names[np.argsort(coef_pos[:-1])[-10:]]

print("Top Keywords for Negative Sentiment:", top_neg)
print("Top Keywords for Positive Sentiment:", top_pos)
print(f"Impact of Review Length: {coef_neg[-1]:.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression

# Adding class_weight='balanced' fixes the "always positive" bias
best_model = LogisticRegression(C=1, solver='liblinear', class_weight='balanced', random_state=42)
best_model.fit(X_train, y_train)

In [2]:
import gradio as gr
import re
from scipy.sparse import hstack

def predict_sentiment(review_text):
    # 1. Clean the input exactly like the training data
    cleaned = review_text.lower()
    cleaned = re.sub(r'<.*?>', '', cleaned)
    cleaned = re.sub(r'[^\w\s]', '', cleaned)

    # 2. Extract the length feature
    word_count = len(cleaned.split())

    # 3. Transform using the TF-IDF and Scaler from previous steps
    # Note: These must exist in your current environment memory
    text_vector = tfidf.transform([cleaned])
    length_scaled = scaler.transform([[word_count]])

    # 4. Combine text and numerical features
    final_features = hstack((text_vector, length_scaled))

    # 5. Predict
    prediction = best_model.predict(final_features)[0]
    probs = best_model.predict_proba(final_features)[0]

    # 6. Map to labels
    labels = ['Negative 😞', 'Neutral 😐', 'Positive 😊']

    # Create a dictionary for the Gradio Label component
    # This shows the percentage for each class
    output_labels = {labels[i]: float(probs[i]) for i in range(3)}

    return output_labels

# Build the improved Interface
app = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(placeholder="Type your review..."),
    outputs=gr.Label(num_top_classes=3, label="Sentiment Confidence"),
    title="Balanced Sentiment Analyzer",
    description="This model is tuned to detect negative sentiment even in imbalanced datasets.",
    examples=[["I hate it, absolute waste of money."], ["Great product!"], ["It was okay."]]
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://18dfc89c8afbb1f896.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
